# L7: Build a Crew to Tailor Job Applications

In this lesson, you will built your first multi-agent system.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import (
  FileReadTool,
  ScrapeWebsiteTool,
  MDXSearchTool,
  SerperDevTool
)

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
read_resume = FileReadTool(file_path='./fake_resume.md')
semantic_search_resume = MDXSearchTool(mdx='./fake_resume.md')

Inserting batches in chromadb: 100%|██████████| 1/1 [00:00<00:00,  2.36it/s]


- Uncomment and run the cell below if you wish to view `fake_resume.md` in the notebook.

In [ ]:
# from IPython.display import Markdown, display
# display(Markdown("./fake_resume.md"))

## Creating Agents

In [5]:
# Agent 1: Researcher
researcher = Agent(
    role="Tech Job Researcher",
    goal="Make sure to do amazing analysis on "
         "job posting to help job applicants",
    tools = [scrape_tool, search_tool],
    verbose=True,
    backstory=(
        "As a Job Researcher, your prowess in "
        "navigating and extracting critical "
        "information from job postings is unmatched."
        "Your skills help pinpoint the necessary "
        "qualifications and skills sought "
        "by employers, forming the foundation for "
        "effective application tailoring."
    )
)

In [6]:
# Agent 2: Profiler
profiler = Agent(
    role="Personal Profiler for Engineers",
    goal="Do increditble research on job applicants "
         "to help them stand out in the job market",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Equipped with analytical prowess, you dissect "
        "and synthesize information "
        "from diverse sources to craft comprehensive "
        "personal and professional profiles, laying the "
        "groundwork for personalized resume enhancements."
    )
)

In [7]:
# Agent 3: Resume Strategist
resume_strategist = Agent(
    role="Resume Strategist for Engineers",
    goal="Find all the best ways to make a "
         "resume stand out in the job market.",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "With a strategic mind and an eye for detail, you "
        "excel at refining resumes to highlight the most "
        "relevant skills and experiences, ensuring they "
        "resonate perfectly with the job's requirements."
    )
)

In [8]:
# Agent 4: Interview Preparer
interview_preparer = Agent(
    role="Engineering Interview Preparer",
    goal="Create interview questions and talking points "
         "based on the resume and job requirements",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Your role is crucial in anticipating the dynamics of "
        "interviews. With your ability to formulate key questions "
        "and talking points, you prepare candidates for success, "
        "ensuring they can confidently address all aspects of the "
        "job they are applying for."
    )
)

## Creating Tasks

In [9]:
# Task for Researcher Agent: Extract Job Requirements
research_task = Task(
    description=(
        "Analyze the job posting URL provided ({job_posting_url}) "
        "to extract key skills, experiences, and qualifications "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of job requirements, including necessary "
        "skills, qualifications, and experiences."
    ),
    agent=researcher,
    async_execution=True
)

In [10]:
# Task for Profiler Agent: Compile Comprehensive Profile
profile_task = Task(
    description=(
        "Compile a detailed personal and professional profile "
        "using the GitHub ({github_url}) URLs, and personal write-up "
        "({personal_writeup}). Utilize tools to extract and "
        "synthesize information from these sources."
    ),
    expected_output=(
        "A comprehensive profile document that includes skills, "
        "project experiences, contributions, interests, and "
        "communication style."
    ),
    agent=profiler,
    async_execution=True
)

- You can pass a list of tasks as `context` to a task.
- The task then takes into account the output of those tasks in its execution.
- The task will not run until it has the output(s) from those tasks.

In [11]:
# Task for Resume Strategist Agent: Align Resume with Job Requirements
resume_strategy_task = Task(
    description=(
        "Using the profile and job requirements obtained from "
        "previous tasks, tailor the resume to highlight the most "
        "relevant areas. Employ tools to adjust and enhance the "
        "resume content. Make sure this is the best resume even but "
        "don't make up any information. Update every section, "
        "inlcuding the initial summary, work experience, skills, "
        "and education. All to better reflrect the candidates "
        "abilities and how it matches the job posting."
    ),
    expected_output=(
        "An updated resume that effectively highlights the candidate's "
        "qualifications and experiences relevant to the job."
    ),
    output_file="tailored_resume.md",
    context=[research_task, profile_task],
    agent=resume_strategist
)

In [12]:
# Task for Interview Preparer Agent: Develop Interview Materials
interview_preparation_task = Task(
    description=(
        "Create a set of potential interview questions and talking "
        "points based on the tailored resume and job requirements. "
        "Utilize tools to generate relevant questions and discussion "
        "points. Make sure to use these question and talking points to "
        "help the candiadte highlight the main points of the resume "
        "and how it matches the job posting."
    ),
    expected_output=(
        "A document containing key questions and talking points "
        "that the candidate should prepare for the initial interview."
    ),
    output_file="interview_materials.md",
    context=[research_task, profile_task, resume_strategy_task],
    agent=interview_preparer
)


## Creating the Crew

In [13]:
job_application_crew = Crew(
    agents=[researcher,
            profiler,
            resume_strategist,
            interview_preparer],

    tasks=[research_task,
           profile_task,
           resume_strategy_task,
           interview_preparation_task],

    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [14]:
job_application_inputs = {
    'job_posting_url': 'https://jobs.lever.co/AIFund/6c82e23e-d954-4dd8-a734-c0c2c5ee00f1?lever-origin=applied&lever-source%5B%5D=AI+Fund',
    'github_url': 'https://github.com/joaomdmoura',
    'personal_writeup': """Noah is an accomplished Software
    Engineering Leader with 18 years of experience, specializing in
    managing remote and in-office teams, and expert in multiple
    programming languages and frameworks. He holds an MBA and a strong
    background in AI and data science. Noah has successfully led
    major tech initiatives and startups, proving his ability to drive
    innovation and growth in the tech industry. Ideal for leadership
    roles that require a strategic and innovative approach."""
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [15]:
### this execution will take a few minutes to run
result = job_application_crew.kickoff(inputs=job_application_inputs)

 [DEBUG]: == Working Agent: Tech Job Researcher
 [INFO]: == Starting Task: Analyze the job posting URL provided (https://jobs.lever.co/AIFund/6c82e23e-d954-4dd8-a734-c0c2c5ee00f1?lever-origin=applied&lever-source%5B%5D=AI+Fund) to extract key skills, experiences, and qualifications required. Use the tools to gather content and identify and categorize the requirements.
 [DEBUG]: == [Tech Job Researcher] Task output: 


 [DEBUG]: == Working Agent: Personal Profiler for Engineers
 [INFO]: == Starting Task: Compile a detailed personal and professional profile using the GitHub (https://github.com/joaomdmoura) URLs, and personal write-up (Noah is an accomplished Software
    Engineering Leader with 18 years of experience, specializing in
    managing remote and in-office teams, and expert in multiple
    programming languages and frameworks. He holds an MBA and a strong
    background in AI and data science. Noah has successfully led
    major tech initiatives and startups, proving his abili

 


Search results: Title: 11 Essential Skills for a Job in Artificial Intelligence
Link: https://ep.jhu.edu/news/11-essential-skills-for-a-job-in-artificial-intelligence/
Snippet: One of the most fundamental skills in AI is proficiency in programming languages. Basic languages to know include Python, R, and Java.
---
Title: What are the key skills and qualifications needed to thrive in the Ai ...
Link: https://www.ziprecruiter.com/e/What-are-the-key-skills-and-qualifications-needed-to-thrive-in-the-Ai-Finance-position-and-why-are-they-important
Snippet: Experience with programming languages such as Python or R, familiarity with financial modeling tools, and certifications like CFA or data analytics credentials ...
---
Title: 7 Key AI Roles and AI Skills You Must Know for Your Career
Link: https://equitably.ai/ai-skills-you-must-have/
Snippet: Hire for T-Shaped Fluency: Combine deep domain expertise (Asset Class Mastery) with Tech Fluency (Prompt Engineering and Data Literacy). The wag

Action: Read website content
Action Input: {"website_url": "https://crewai.com"} 

I tried reusing the same input, I must stop using this action input. I'll try something else instead.



Thought: The search results provide various articles related to AI skills and qualifications, but not the specific job posting details. I need to delegate the task to a co-worker who specializes in analyzing job requirements.

Action: Delegate work to co-worker
Action Input: {"coworker": "Personal Profiler for Engineers", "task": "Analyze the job posting requirements", "context": "Need to extract key skills, experiences, and qualifications required for a job posting at AI Fund. The website content could not be accessed, and the search results did not provide the specific details."}

> Entering new CrewAgentExecutor chain...
Action: Search the internet
Action Input: {"search_query": "Noah Software Engineering Leader managing remote and in-office teams AI data science tech initiatives startups"}I need t

Thought: I have gathered information from both the GitHub profile and personal write-up for Noah.

Action: Search the internet
Action Input: {"search_query": "Noah Shapiro Software Engineering Leader managing remote and in-office teams AI data science tech initiatives startups"}Final Answer:
Noah Williams is a distinguished Software Engineering Leader with an 18-year tenure in the technology industry. He has excelled in leading remote and in-office engineering teams, with expertise in software development, process innovation, team collaboration, and proficiency in programming languages such as Ruby, Python, JavaScript, TypeScript, and Elixir. Noah has significant experience in data science and machine learning, spearheading successful deployments of scalable AI solutions and innovative data model development. 

In his work history, Noah has held positions such as Director of Software Engineering, Senior Software Engineering Manager, Founder & CEO of InnovPet, Engineering Manager at Eli

Final Answer:

# Noah Williams
- Email: noah.williams@example.dev
- Phone: +44 11 111 11111

## Profile
Noah Williams is an accomplished Software Engineering Leader with 18 years of experience, specializing in managing remote and in-office teams. He is an expert in multiple programming languages and frameworks, holding an MBA and having a strong background in AI and data science. Noah has successfully led major tech initiatives and startups, demonstrating his ability to drive innovation and growth in the tech industry. Ideal for leadership roles that require a strategic and innovative approach.

## Work History

### DataKernel: Director of Software Engineering (remote) — 2022 - Present
- Transformed the engineering division into a key revenue pillar for DataKernel, rapidly expanding the customer base and enhancing product capabilities.
- Spearheaded integration of cutting-edge AI technologies and scalable vector databases, leading to significant market positioning.
- Focused on skill d

Final Answer:
Based on the tailored resume of Noah Williams and the job requirements, here are some potential interview questions and talking points to help Noah showcase his qualifications and experiences effectively:

1. Can you walk us through a specific project where you successfully integrated cutting-edge AI technologies and scalable databases to enhance product capabilities and market positioning?
2. How have you approached skill development within your engineering teams to achieve strategic project goals and influence company direction?
3. As a founder and CEO of InnovPet, can you discuss the challenges you faced in overseeing product development and navigating successful market entry for innovative IoT solutions?
4. In your role as an Engineering Manager at EliteDevs, how did you foster a culture of productivity and innovation within the multiple engineering teams you managed?
5. Can you provide an example of a strategic initiative you led at PrintPack that resulted in a signi

- Dislplay the generated `tailored_resume.md` file.

In [16]:
from IPython.display import Markdown, display
display(Markdown("./tailored_resume.md"))

# Noah Williams
- Email: noah.williams@example.dev
- Phone: +44 11 111 11111

## Profile
Noah Williams is an accomplished Software Engineering Leader with 18 years of experience, specializing in managing remote and in-office teams. He is an expert in multiple programming languages and frameworks, holding an MBA and having a strong background in AI and data science. Noah has successfully led major tech initiatives and startups, demonstrating his ability to drive innovation and growth in the tech industry. Ideal for leadership roles that require a strategic and innovative approach.

## Work History

### DataKernel: Director of Software Engineering (remote) — 2022 - Present
- Transformed the engineering division into a key revenue pillar for DataKernel, rapidly expanding the customer base and enhancing product capabilities.
- Spearheaded integration of cutting-edge AI technologies and scalable vector databases, leading to significant market positioning.
- Focused on skill development and strategic project goals, influencing company direction.

### DataKernel: Senior Software Engineering Manager (remote) — 2019 - 2022
- Directed engineering strategy and operations, shaping the company's technological trajectory.
- Managed diverse teams across multiple time zones, fostering transparency and mutual respect.
- Recruited, mentored, and retained top talent, fostering continuous improvement and high performance.

### InnovPet: Founder & CEO (remote) — 2019 - 2022
- Founded InnovPet, a startup focused on innovative IoT solutions for pet care.
- Oversaw product development and successful market entry.
- Established advisory board, production facilities overseas, and navigated successful funding phase.

### EliteDevs: Engineering Manager (remote) — 2018 - 2019
- Formulated and executed strategic plans, enhancing inter-departmental coordination and trust.
- Managed multiple engineering teams, fostering a culture of productivity and innovation.
- Implemented goal-setting frameworks aligned with company's long-term goals.

### PrintPack: Engineering Manager (remote) — 2016 - 2018
- Led formation of high-performance engineering team, increasing company revenue by 500%.
- Integrated data analytics into decision-making processes, revolutionizing customer behavior analysis.

### DriveAI: Senior Software Engineer (remote) — 2015 - 2016
- Developed and optimized a central API, improving functionality for large engineering teams and users.
- Implemented critical enhancements, including advanced caching strategies for improved system performance.

### BetCraft: CTO — 2013 - 2015
- Led technology strategy post-Series A funding, guiding company through significant technological advancement.
- Improved platform performance and expanded market reach through strategic initiatives and partnerships.

## Education

### MBA in Information Technology
London Business School - MBA

### Advanced Leadership Techniques
University of London - Certification

### Data Science Specialization
Coursera (Johns Hopkins University) - Certification

### B.Sc. in Computer Science
University of Edinburgh - Bachelor’s degree

Noah Williams is an ideal candidate for senior executive roles, with a robust blend of technical and strategic expertise.

- Dislplay the generated `interview_materials.md` file.

In [ ]:
display(Markdown("./interview_materials.md"))

# CONGRATULATIONS!!!

## Share your accomplishment!
- Once you finish watching all the videos, you will see the "In progress" image on the bottom left turn into "Accomplished".
- Click on "Accomplished" to view the course completion page with your name on it.
- Take a screenshot and share on LinkedIn, X (Twitter), or Facebook.  
- **Tag @Joāo (Joe) Moura, @crewAI, and @DeepLearning.AI, (and a few of your friends if you'd like them to try out the course)**
- **Joāo and DeepLearning.AI will "like"/reshare/comment on your post!**

## Get a completion badge that you can add to your LinkedIn profile!
- Go to [learn.crewai.com](https://learn.crewai.com).
- Upload your screenshot of your course completion page.
- You'll get a badge from CrewAI that you can share!

(Joāo will also talk about this in the last video of the course.)